# SET UP PREDICTION

**IMPORTANTE**: Recuerda hacer una copia de esta plantilla para no machacar la original.

## 1. CREAR EL ENTORNO PARA EL PROYECTO

Crear un nuevo entorno e instalar los paquetes básicos:

**Hemos adaptado algunas versiones de librerías para que no te den fallo de compatibilidad** 

1. Ir al prompt o terminal
2. Copiar y poner la línea inferior:

conda create --name 007_airbnb -c conda-forge python=3.11.11 numpy pandas matplotlib seaborn scikit-learn=1.3.1 scipy=1.9.3 sqlalchemy xgboost=2.0.3 jupyter

3. Activar el nuevo entorno: conda activate 007_airbnb

4. Instalar los paquetes que están en otros canales:

conda install -c conda-forge pyjanitor scikit-plot yellowbrick imbalanced-learn jupyter_contrib_nbextensions cloudpickle jupyter_client folium

conda install -c districtdatalabs yellowbrick

pip install category_encoders==2.6.2

5. Crear el environment.yml: conda env export > 007_airbnb.yml

*Nota: es posible que te lo haya creado en C:\Users\tu_usuario, más adelante moveremos este archivo al directorio del proyecto*

En el Dell los crea en: C:\Users\Oscar

**Saber la versión de los paquetes instalados en un entorno:**
* Activamos el entorno: conda activate 007_airbnb
* Saber las versiones de todos los paquetes: conda list
* Saber las versiones de un paquete concreto: conda list matplotlib

## 2. IMPORTAR PAQUETES

In [1]:
#Posibles paquetes necesarios para el desarrollo del proyecto. No harán falta todos siempre. Seleccionar los necesarios
import os
import numpy as np
import pandas as pd
import pickle  
import cloudpickle #Mismas funciones de pickle de guardar modelos + funciones de guardado de las funciones específicasimport seaborn as sns
import sqlalchemy as sa
import scipy as sp
import matplotlib.pyplot as plt
%matplotlib inline
%config IPCompleter.greedy = True

#Formato sin notación científica
pd.options.display.float_format = '{:15.2f}'.format 

#Automcompletar rápido
%config IPCompleter.greedy=True

#Desactivar la notación científica
pd.options.display.float_format = '{:.2f}'.format

#Desactivar los warnings
import warnings
warnings.filterwarnings("ignore")

#Mostrar el máximo de filas posibles de una tabla
pd.set_option('display.max_rows', 100) #Número de filas que deben verse. None = Máx

#Mostrar mas caracteres de las columnas. Se usa cuando se corta el texto
pd.set_option('display.max_colwidth', None) #Número de caractres que deben verse. None = Máx

#Representación visual de un pipeline
from sklearn import set_config
set_config(display = 'diagram') #diagram/text 

#Transformación de variables
from janitor import clean_names
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import TargetEncoder
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.preprocessing import Binarizer
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import QuantileTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import MaxAbsScaler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler

#Modelos ML: CLASIFICACIÓN
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
#Métrica de error: Habrá que incluir la que necesitemos
from sklearn.metrics import roc_auc_score

#Modelos ML: REGRESIÓN
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
#Métrica de error: Habrá que incluir la que necesitemos
from sklearn.metrics import mean_absolute_error

#Modelos ML: NO SUPERVISADO
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics import calinski_harabasz_score
from sklearn.metrics import davies_bouldin_score

## 3. CREAR EL DIRECTORIO DEL PROYECTO

In [2]:
#Ubicamos el proyecto y el nombre 
raiz = 'C:/Users/Oscar/OneDrive - FM4/Escritorio/Python Data Mastery/EstructuraDirectorio/03_MACHINE_LEARNING/'
nombre_carpeta = '08_CASOS/'
nombre_proyecto = '007_AIRBNB'
path = raiz + nombre_carpeta + nombre_proyecto

In [3]:
#Definimos la estructura de carpetas y la jerarquía que creará. Si quisieramos mas jerarquía, la pondríamos como carpeta 
try:
    os.makedirs(path)
    os.makedirs(path + '/01_Documentos')
    os.makedirs(path + '/02_Datos')
    os.makedirs(path + '/02_Datos/01_Originales')
    os.makedirs(path + '/02_Datos/02_Validacion')
    os.makedirs(path + '/02_Datos/03_Trabajo')
    os.makedirs(path + '/02_Datos/04_Caches')
    os.makedirs(path + '/03_Notebooks')
    os.makedirs(path + '/03_Notebooks/01_Funciones')
    os.makedirs(path + '/03_Notebooks/02_Desarrollo')
    os.makedirs(path + '/03_Notebooks/03_Sistema')
    os.makedirs(path + '/04_Modelos')
    os.makedirs(path + '/05_Resultados')
    os.makedirs(path + '/09_Otros')
    
except OSError:
    print ("La creación del directorio %s ha fallado" % path)
else:
    print ("Se ha creado satisfactoriamente el directorio %s " % path)

#Situar el directorio de trabajo en el proyecto
os.chdir(path)

#Buscar el archivo del entorno del proyecto que tendrá el nombre definido arriba (termina en .yml) y moverlo
#de la ubicación del ordenador C:\Users\tu_usuario a  la carpeta del proyecto /01_Documentos

La creación del directorio C:/Users/Oscar/OneDrive - FM4/Escritorio/Python Data Mastery/EstructuraDirectorio/03_MACHINE_LEARNING/08_CASOS/007_AIRBNB ha fallado


## 4. CREAR LOS DATASETS INICIALES

In [ ]:
#Crear la conexión a la base de datos
con = sa.create_engine('sqlite:///../Datos/airbnb2025.db')

#Conseguir la lista de las tablas de 'con'
from sqlalchemy import inspect
insp = inspect(con)
tablas = insp.get_table_names()
tablas

#Cargar las tablas individualmente
listings = pd.read_sql('listings', con)
listings_det = pd.read_sql('listings_det', con)

#Cargar todas las tablas de 'con' al mismo tiempo
for tabla in tablas:
    exec(f'{tabla} = pd.read_sql(tabla, con)')

#Extraer las dimensiones de todas las tablas de 'con' al mismo tiempo
for cada in tablas:
    print(cada + ': ' + str(eval(cada).shape))

Poner los datos originales en la carpeta '/02_Datos/01_Originales'

### 4.1. Importar los datos

Sustituir el nombre_fichero_datos.

In [4]:
nombre_fichero_datos = 'airbnb2025.db'
ruta_completa = path + '/02_Datos/01_Originales/' + nombre_fichero_datos

Cargar los datos.

In [5]:
#Función carga de LOS NOMBRES de todas las tablas de la base de datos
import sqlalchemy as sa
con = sa.create_engine('sqlite:///' +  ruta_completa)

from sqlalchemy import inspect
inspector = inspect(con)
tablas = inspector.get_table_names()
tablas  

['calendar',
 'cat',
 'df_preparado',
 'listings',
 'listings_det',
 'neigh',
 'num',
 'precio_m2',
 'target']

In [6]:
#Cargar todas las tablas de 'con' al mismo tiempo
for tabla in tablas:
    exec(f'{tabla} = pd.read_sql(tabla, con)')

In [7]:
datos = df_preparado.copy()
datos.head()

,index,id,name,host_response_time,host_is_superhost,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood_cleansed,property_type,...,precio_total,ocupacion,bedrooms_disc,accommodates_disc,beds_disc,number_of_reviews_disc,m2,precio_compra,pdi_sol,pisos_rentables
0,0,21853,Bright and airy room,no response,f,"['email', 'phone']",t,t,Cármenes,Private room in rental unit,...,31.00,60,01_Una,0-3,1,5-71,50,110880.00,3.52,1
1,1,30320,Great Vacational Apartments,no response,f,"['email', 'phone']",t,f,Sol,Entire rental unit,...,149.00,100,01_Una,0-3,1,71-1136,50,246800.00,0.23,1
2,2,30959,Beautiful loft in Madrid Center,no response,f,"['email', 'phone']",t,f,Embajadores,Entire loft,...,92.00,100,01_Una,0-3,1,1-4,50,246800.00,0.50,1
3,3,40916,Holiday Apartment Madrid Center,no response,f,"['email', 'phone']",t,f,Universidad,Entire rental unit,...,124.00,100,01_Una,0-3,3,5-71,50,246800.00,0.67,1
4,4,62423,MAGIC ARTISTIC HOUSE IN THE CENTER OF MADRID,within an hour,f,"['email', 'phone']",t,t,Justicia,Private room in rental unit,...,137.62,9,01_Una,4,3,71-1136,50,246800.00,0.62,0


In [8]:
#Vamos a eliminar las variables que hemos usado para generar la target ya que hemos terminado el proyecto de Discovery

#Tras pasar por la fase de transformación de datos del proyecto de prediction, vuelvo a eliminar 'neighbourhood_cleansed'
#porque genera mucho ruido en el avance y no aporta gran información por encima de 'neighbourhood_group'
datos = datos.drop(columns=['precio_total','ocupacion','precio_compra','neighbourhood_cleansed'])
datos.head()

,index,id,name,host_response_time,host_is_superhost,host_verifications,host_has_profile_pic,host_identity_verified,property_type,room_type,...,calculated_host_listings_count_shared_rooms,reviews_per_month,precio_m2,bedrooms_disc,accommodates_disc,beds_disc,number_of_reviews_disc,m2,pdi_sol,pisos_rentables
0,0,21853,Bright and airy room,no response,f,"['email', 'phone']",t,t,Private room in rental unit,Private room,...,0,0.27,2772,01_Una,0-3,1,5-71,50,3.52,1
1,1,30320,Great Vacational Apartments,no response,f,"['email', 'phone']",t,f,Entire rental unit,Entire home/apt,...,0,0.98,6170,01_Una,0-3,1,71-1136,50,0.23,1
2,2,30959,Beautiful loft in Madrid Center,no response,f,"['email', 'phone']",t,f,Entire loft,Entire home/apt,...,0,0.07,6170,01_Una,0-3,1,1-4,50,0.50,1
3,3,40916,Holiday Apartment Madrid Center,no response,f,"['email', 'phone']",t,f,Entire rental unit,Entire home/apt,...,0,0.29,6170,01_Una,0-3,3,5-71,50,0.67,1
4,4,62423,MAGIC ARTISTIC HOUSE IN THE CENTER OF MADRID,within an hour,f,"['email', 'phone']",t,t,Private room in rental unit,Private room,...,0,2.73,6170,01_Una,4,3,71-1136,50,0.62,0


### 4.2. Extraer y reservar el dataset de validación

In [9]:
val= datos.sample(frac = 0.3)

In [10]:
nombre_fichero_validacion = 'validacion.csv'
ruta_completa = path + '/02_Datos/02_Validacion/' + nombre_fichero_validacion
val.to_csv(ruta_completa)

### 4.3. Extraer y guardar el dataset de trabajo

In [11]:
trabajo = datos.loc[~datos.index.isin(val.index)]

In [12]:
nombre_fichero_trabajo = 'trabajo.csv'
ruta_completa = path + '/02_Datos/03_Trabajo/' + nombre_fichero_trabajo
trabajo.to_csv(ruta_completa)

In [13]:
val.shape

(7774, 59)

In [14]:
trabajo.shape

(18139, 59)

### 4.4. Extraer y guardar una muestra (opcional)

Actualizar el tamaño de la muestra.

In [15]:
#muestra= trabajo.sample(n = 20000)

In [16]:
#nombre_fichero_muestra = 'muestra.csv'
#ruta_completa = path + '/02_Datos/03_Trabajo/' + nombre_fichero_muestra
#muestra.to_csv(ruta_completa)